In [4]:
import os
import numpy as np
import matplotlib.pyplot as plt
import scipy.ndimage
import meep as mp

# ==============================================================================
# 共用固定參數 (與 Design Region 大小無關的設定)
# ==============================================================================
RESOLUTION = 40
DPML = 1.0
WG_LENGTH = 1.0
WG_WIDTH = 1.0
N_SIO2 = 1.44
N_SI = 3.48
SIO2_MEDIUM = mp.Medium(index=N_SIO2)
SI_MEDIUM = mp.Medium(index=N_SI)

wl_cen = 1.55 
FCEN = 1 / wl_cen
DF = 0.1 * FCEN
TARGET_WLS = [1.52, 1.56, 1.60]
TARGET_FREQS = [1/wl for wl in TARGET_WLS]

KERNEL_SIZE = 5
KERNEL_SIGMA = 1.00
TANH_BETA = 50
TANH_ETA = 0.5

# ==============================================================================
# 幾何與矩陣處理函數
# ==============================================================================
def gaussian_kernel(size=KERNEL_SIZE, sigma=KERNEL_SIGMA):
    ax = np.arange(-(size//2), size//2 + 1)
    xx, yy = np.meshgrid(ax, ax)
    kernel = np.exp(-(xx**2 + yy**2) / (2.0 * sigma**2))
    kernel /= np.sum(kernel)
    return kernel

def tanh_projection(x, beta=TANH_BETA, eta=TANH_ETA):
    num = np.tanh(beta * eta) + np.tanh(beta * (x - eta))
    den = np.tanh(beta * eta) + np.tanh(beta * (1 - eta))
    return num / den

def get_projected_density_from_2d(grid_matrix):
    """直接從讀取的 2D 矩陣進行卷積與投影"""
    grid_rows, grid_cols = grid_matrix.shape
    kernel = gaussian_kernel(size=KERNEL_SIZE, sigma=KERNEL_SIGMA)
    try:
        from scipy.signal import convolve2d
        density = convolve2d(grid_matrix, kernel, mode='same', boundary='symm')
    except Exception:
        k = kernel.shape[0]
        pad = k // 2
        img_p = np.pad(grid_matrix, ((pad, pad), (pad, pad)), mode='reflect')
        density = np.zeros_like(grid_matrix)
        for i in range(grid_rows):
            for j in range(grid_cols):
                density[i, j] = np.sum(img_p[i:i+k, j:j+k] * kernel)
                
    projected_density = tanh_projection(density, beta=TANH_BETA, eta=TANH_ETA)
    return np.clip(projected_density, 0.0, 1.0)

# ==============================================================================
# 動態 MEEP 模擬核心
# ==============================================================================
def evaluate_specific_config(npy_path, mdm_lc):
    print(f"\n>>> 正在處理: {npy_path} (Design Region: {mdm_lc} μm)")
    
    # 1. 讀取並還原 config 矩陣
    loaded_config = np.load(npy_path)
    grid_matrix = loaded_config.T 
    
    # 【關鍵修正】：直接從讀取的矩陣取得展開後的完整維度 (e.g. 30x30)
    grid_rows, grid_cols = grid_matrix.shape
    print(f"  -- 讀取到矩陣維度: {grid_rows} x {grid_cols} --")
    
    # 計算平滑投影
    density = get_projected_density_from_2d(grid_matrix)
    weights = density.flatten()
    
    # 2. 動態計算邊界與空間
    sx = 2 * DPML + mdm_lc + 2 * WG_LENGTH
    sy = 2 * DPML + mdm_lc + 2 * WG_LENGTH
    cell = mp.Vector3(sx, sy, 0)
    
    # 這裡的 grid_cols 和 grid_rows 現在會正確對應到 weights 的長度
    material_grid = mp.MaterialGrid(mp.Vector3(grid_cols, grid_rows), SIO2_MEDIUM, SI_MEDIUM, weights=weights)
    material_grid.smoothing_radius = 1.0 
    mdm_structure = [mp.Block(size=mp.Vector3(mdm_lc, mdm_lc, mp.inf), center=mp.Vector3(), material=material_grid)]
    
    input_wg_center_x = -mdm_lc / 2 - (WG_LENGTH + DPML) / 2    
    through_wg_center_x = mdm_lc / 2 + (WG_LENGTH + DPML) / 2   
    cross_top_center_y = mdm_lc / 2 + (WG_LENGTH + DPML) / 2    
    cross_bot_center_y = -mdm_lc / 2 - (WG_LENGTH + DPML) / 2   
    
    fixed_geometry = [
        mp.Block(size=mp.Vector3(WG_LENGTH + DPML + 0.1, WG_WIDTH, mp.inf), center=mp.Vector3(input_wg_center_x, 0), material=SI_MEDIUM),
        mp.Block(size=mp.Vector3(WG_LENGTH + DPML + 0.1, WG_WIDTH, mp.inf), center=mp.Vector3(through_wg_center_x, 0), material=SI_MEDIUM),
        mp.Block(size=mp.Vector3(WG_WIDTH, WG_LENGTH + DPML + 0.1, mp.inf), center=mp.Vector3(0, cross_top_center_y), material=SI_MEDIUM),
        mp.Block(size=mp.Vector3(WG_WIDTH, WG_LENGTH + DPML + 0.1, mp.inf), center=mp.Vector3(0, cross_bot_center_y), material=SI_MEDIUM),
    ]
    full_geometry = fixed_geometry + mdm_structure
    
    src_center = mp.Vector3(-sx / 2 + DPML + 0.2, 0)
    src_size = mp.Vector3(0, WG_WIDTH) 
    mon_x_through = mdm_lc / 2 + WG_LENGTH / 2
    monitor_size_y = mp.Vector3(0, WG_WIDTH * 3)
    
    results = {'TE0': [], 'TE1': []}
    modes = ['TE0', 'TE1']
    
    for mode_name in modes:
        mp.Simulation(cell_size=cell, resolution=1, boundary_layers=[]).reset_meep()
        print(f"  -- 正在模擬 {mode_name} 模態 --")
        
        props = {'TE0': {'band': 1, 'parity': mp.EVEN_Y}, 'TE1': {'band': 2, 'parity': mp.ODD_Y}}[mode_name]
        
        sources = [mp.EigenModeSource(src=mp.GaussianSource(FCEN, fwidth=DF), 
                                      center=src_center, size=src_size, direction=mp.X, 
                                      eig_band=props['band'], eig_parity=props['parity'])]
        
        sim = mp.Simulation(cell_size=cell, boundary_layers=[mp.PML(DPML)], 
                            geometry=full_geometry, sources=sources, 
                            resolution=RESOLUTION, default_material=SIO2_MEDIUM)
        
        # 【關鍵修正】：為了相容所有 MEEP 版本，我們用你最原始的陣列寫法，擴增到 100 個波長點
        nfreq = 100
        # 直接鎖定你要的波長範圍 1.52 到 1.60
        target_wls = np.linspace(1.52, 1.60, nfreq) 
        target_freqs = [1/wl for wl in target_wls]
        
        in_flux_region = mp.ModeRegion(center=mp.Vector3(src_center.x + 0.5, 0), size=monitor_size_y)
        through_flux_region = mp.ModeRegion(center=mp.Vector3(mon_x_through, 0), size=monitor_size_y)
        
        # 針對 100 個頻率點，各自建立獨立的 Monitor 
        norm_fluxes = [sim.add_mode_monitor(f, 0, 1, in_flux_region) for f in target_freqs]
        flux_throughs = [sim.add_mode_monitor(f, 0, 1, through_flux_region) for f in target_freqs]
        
        # 執行模擬
        sim.run(until_after_sources=mp.stop_when_fields_decayed(50, mp.Ez, mp.Vector3(mon_x_through, WG_WIDTH/4), 1e-4))
        
        wls = []
        trans_dbs = []
        
        # 迴圈抓取這 100 個獨立的 Monitor 資料
        for i in range(nfreq):
            wl = target_wls[i]
            
            res_input = sim.get_eigenmode_coefficients(norm_fluxes[i], [props['band']], eig_parity=props['parity']).alpha[0, 0, 0]
            input_power = np.abs(res_input)**2 + 1e-12
            
            through_coeff = sim.get_eigenmode_coefficients(flux_throughs[i], [props['band']], eig_parity=props['parity']).alpha[0, 0, 0]
            trans = np.abs(through_coeff)**2 / input_power
            trans_db = 10 * np.log10(trans + 1e-9)
            
            wls.append(wl)
            trans_dbs.append(trans_db)
            
        results[mode_name] = {'wls': wls, 'trans_dbs': trans_dbs}
        sim.reset_meep()
        
    return results

# ==============================================================================
# 繪圖腳本 (TE0 與 TE1 分開繪製)
# ==============================================================================
def plot_separated_spectra(all_results):
    # 不需要 scipy.interpolate 了
    
    colors = ['#1f77b4', '#d62728', '#2ca02c'] 
    modes = ['TE0', 'TE1']
    
    for mode in modes:
        plt.figure(figsize=(6, 5)) 
        
        for idx, exp in enumerate(all_results):
            label_prefix = exp['label']
            color = colors[idx % len(colors)]
            
            # 讀取剛剛存好的 100 個點
            wls = np.array(exp['results'][mode]['wls'])
            trans_dbs = np.array(exp['results'][mode]['trans_dbs'])
            
            # 因為 MEEP 回傳的頻率是從小到大 (波長是大到小)，我們先排序一下確保畫圖線條順暢
            sort_idx = np.argsort(wls)
            wls = wls[sort_idx]
            trans_dbs = trans_dbs[sort_idx]
            
            # 直接把真實物理點連成曲線！
            plt.plot(wls, trans_dbs, color=color, linestyle='-', 
                     linewidth=2.0, label=label_prefix)

        plt.title(f"Transmission Spectrum ({mode})", fontsize=16, fontweight='bold')
        plt.xlabel(r"Wavelength ($\mu$m)", fontsize=14)
        plt.ylabel("Transmission (dB)", fontsize=14)
        
        # 為了跟文獻比較，你可以把 Y 軸設得更寬一點，例如 -4 到 0
        plt.ylim([-4.0, 0.0]) 
        
        # 限制 X 軸顯示範圍，對齊文獻的 1.52 到 1.60
        plt.xlim([1.52, 1.60])
        
        plt.grid(True, alpha=0.4, linestyle='--')
        plt.legend(loc='lower left', fontsize=12) # 移到左下角避免擋住右邊通常比較高的穿透率
        plt.tight_layout()
        
        filename = f"Continuous_Spectrum_{mode}.png"
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"✅ 已生成並儲存連續頻譜圖表: {filename}")

if __name__ == "__main__":
    # 在這裡配置你的三個實驗資料路徑與參數
    experiments = [
        {
            'label': 'Design Region 1 (2.4 μm)',
            'path': '/home/oscar3102/FMQA/Ultra_compact_waveguide_Crossing/Results_Crossing/Crossing_waveguide_1000_100x30_20260531_015644/best_config_30x30.npy',
            'mdm_lc': 2.4,
            'rows': 15, # 保留作為紀錄，程式會自動從 npy 抓取實際的 30x30
            'cols': 15
        },
        {
            'label': 'Design Region 2 (5.0 μm)',
            'path': '/home/oscar3102/FMQA/Ultra_compact_waveguide_Crossing/Results_Crossing/Crossing_waveguide_1000_100x30_20260529_220817/best_config_20x20.npy',
            'mdm_lc': 5.0, 
            'rows': 10, # 保留作為紀錄，程式會自動從 npy 抓取實際的 20x20
            'cols': 10
        },
        {
            'label': 'Design Region 3 (4.0 μm)',
            'path': '/home/oscar3102/FMQA/Ultra_compact_waveguide_Crossing/Results_Crossing/Crossing_waveguide_1000_100x30_20260531_001805/best_config_30x30.npy',
            'mdm_lc': 4.0, 
            'rows': 15, # 保留作為紀錄，程式會自動從 npy 抓取實際的 30x30
            'cols': 15
        }
    ]
    
    all_results = []
    
    for exp in experiments:
        if os.path.exists(exp['path']):
            # 【關鍵修正】：呼叫時只傳入 path 和 mdm_lc，讓函數自己去讀取正確的矩陣形狀
            res = evaluate_specific_config(exp['path'], exp['mdm_lc'])
            all_results.append({
                'label': exp['label'],
                'results': res
            })
        else:
            print(f"找不到檔案: {exp['path']}")

    if all_results:
        plot_separated_spectra(all_results)
        print("\n🎉 所有圖表繪製完成！")


>>> 正在處理: /home/oscar3102/FMQA/Ultra_compact_waveguide_Crossing/Results_Crossing/Crossing_waveguide_1000_100x30_20260531_015644/best_config_30x30.npy (Design Region: 2.4 μm)
  -- 讀取到矩陣維度: 30 x 30 --
  -- 正在模擬 TE0 模態 --
-----------
Initializing structure...
time for choose_chunkdivision = 0.000291109 s
Working in 2D dimensions.
Computational cell is 6.4 x 6.4 x 0 with resolution 40
     block, center = (-2.2,0,0)
          size (2.1,1,1e+20)
          axes (1,0,0), (0,1,0), (0,0,1)
          dielectric constant epsilon diagonal = (12.1104,12.1104,12.1104)
     block, center = (2.2,0,0)
          size (2.1,1,1e+20)
          axes (1,0,0), (0,1,0), (0,0,1)
          dielectric constant epsilon diagonal = (12.1104,12.1104,12.1104)
     block, center = (0,2.2,0)
          size (1,2.1,1e+20)
          axes (1,0,0), (0,1,0), (0,0,1)
          dielectric constant epsilon diagonal = (12.1104,12.1104,12.1104)
     block, center = (0,-2.2,0)
          size (1,2.1,1e+20)
          axes (1,0,0), (

time for set_epsilon = 0.133245 s
-----------
MPB solved for frequency_1(2.24516,0,0) = 0.648359 after 17 iters
MPB solved for frequency_1(2.23407,0,0) = 0.645161 after 7 iters
MPB solved for frequency_1(2.23407,0,0) = 0.645161 after 1 iters
field decay(t = 50.0125): 0.0011371151380065562 / 0.0011371151380065562 = 1.0
on time step 6552 (time=81.9), 0.000610557 s/step
field decay(t = 100.025): 1.1694663930929565 / 1.1694663930929565 = 1.0
field decay(t = 150.0375): 0.8028260598347577 / 1.1694663930929565 = 0.6864892095885513
on time step 13157 (time=164.463), 0.000605736 s/step
field decay(t = 200.05): 1.0063738948350495e-06 / 1.1694663930929565 = 8.605410987257473e-07
run 0 finished at t = 200.05 (16004 timesteps)
MPB solved for frequency_1(2.28947,0,0) = 0.669811 after 24 iters
MPB solved for frequency_1(2.24746,0,0) = 0.657897 after 7 iters
MPB solved for frequency_1(2.24745,0,0) = 0.657895 after 1 iters
Dominant planewave for band 1: (2.247451,-0.000000,0.000000)
MPB solved for freq

In [5]:
import os
import numpy as np
import matplotlib.pyplot as plt
import meep as mp

# ==============================================================================
# 共用固定參數
# ==============================================================================
RESOLUTION = 40
DPML = 1.0
WG_LENGTH = 1.0
WG_WIDTH = 1.0
N_SIO2 = 1.44
N_SI = 3.48
SIO2_MEDIUM = mp.Medium(index=N_SIO2)
SI_MEDIUM = mp.Medium(index=N_SI)

wl_cen = 1.55 
FCEN = 1 / wl_cen
DF = 0.1 * FCEN

KERNEL_SIZE = 5
KERNEL_SIGMA = 1.00
TANH_BETA = 50
TANH_ETA = 0.5

# ==============================================================================
# 幾何與矩陣處理函數
# ==============================================================================
def gaussian_kernel(size=KERNEL_SIZE, sigma=KERNEL_SIGMA):
    ax = np.arange(-(size//2), size//2 + 1)
    xx, yy = np.meshgrid(ax, ax)
    kernel = np.exp(-(xx**2 + yy**2) / (2.0 * sigma**2))
    kernel /= np.sum(kernel)
    return kernel

def tanh_projection(x, beta=TANH_BETA, eta=TANH_ETA):
    num = np.tanh(beta * eta) + np.tanh(beta * (x - eta))
    den = np.tanh(beta * eta) + np.tanh(beta * (1 - eta))
    return num / den

def get_projected_density_from_2d(grid_matrix):
    grid_rows, grid_cols = grid_matrix.shape
    kernel = gaussian_kernel(size=KERNEL_SIZE, sigma=KERNEL_SIGMA)
    try:
        from scipy.signal import convolve2d
        density = convolve2d(grid_matrix, kernel, mode='same', boundary='symm')
    except Exception:
        k = kernel.shape[0]
        pad = k // 2
        img_p = np.pad(grid_matrix, ((pad, pad), (pad, pad)), mode='reflect')
        density = np.zeros_like(grid_matrix)
        for i in range(grid_rows):
            for j in range(grid_cols):
                density[i, j] = np.sum(img_p[i:i+k, j:j+k] * kernel)
                
    projected_density = tanh_projection(density, beta=TANH_BETA, eta=TANH_ETA)
    return np.clip(projected_density, 0.0, 1.0)

# ==============================================================================
# 繪圖函數 (單張繪製)
# ==============================================================================
def plot_single_continuous_spectrum(wls, trans_dbs, mode_name, exp_label, color_hex):
    plt.figure(figsize=(6, 5)) 
    
    wls = np.array(wls)
    trans_dbs = np.array(trans_dbs)
    sort_idx = np.argsort(wls)
    wls = wls[sort_idx]
    trans_dbs = trans_dbs[sort_idx]
    
    plt.plot(wls, trans_dbs, color=color_hex, linestyle='-', linewidth=2.5, label='FMQA')

    # 標題加入實驗名稱，方便辨識
    plt.title(f"{exp_label} - {mode_name}", fontsize=16, fontweight='bold')
    plt.xlabel(r"Wavelength ($\mu$m)", fontsize=14)
    plt.ylabel("Transmission (dB)", fontsize=14)
    
    plt.ylim([-4.0, 0.0]) 
    plt.xlim([1.52, 1.60])
    
    plt.grid(True, alpha=0.4, linestyle='--')
    plt.legend(loc='lower left', fontsize=12)
    plt.tight_layout()
    
    # 處理檔名，移除空格和括號避免系統問題
    safe_label = exp_label.replace(" ", "_").replace("(", "").replace(")", "").replace("μm", "um")
    filename = f"Spectrum_{safe_label}_{mode_name}.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ 已儲存圖表: {filename}")

# ==============================================================================
# 動態 MEEP 模擬核心 (100個頻率點)
# ==============================================================================
def evaluate_and_plot(npy_path, mdm_lc, exp_label, color_hex):
    print(f"\n==================================================")
    print(f"  開始處理: {exp_label}")
    print(f"==================================================")
    
    loaded_config = np.load(npy_path)
    grid_matrix = loaded_config.T 
    grid_rows, grid_cols = grid_matrix.shape
    
    density = get_projected_density_from_2d(grid_matrix)
    weights = density.flatten()
    
    sx = 2 * DPML + mdm_lc + 2 * WG_LENGTH
    sy = 2 * DPML + mdm_lc + 2 * WG_LENGTH
    cell = mp.Vector3(sx, sy, 0)
    
    material_grid = mp.MaterialGrid(mp.Vector3(grid_cols, grid_rows), SIO2_MEDIUM, SI_MEDIUM, weights=weights)
    material_grid.smoothing_radius = 1.0 
    mdm_structure = [mp.Block(size=mp.Vector3(mdm_lc, mdm_lc, mp.inf), center=mp.Vector3(), material=material_grid)]
    
    input_wg_center_x = -mdm_lc / 2 - (WG_LENGTH + DPML) / 2    
    through_wg_center_x = mdm_lc / 2 + (WG_LENGTH + DPML) / 2   
    cross_top_center_y = mdm_lc / 2 + (WG_LENGTH + DPML) / 2    
    cross_bot_center_y = -mdm_lc / 2 - (WG_LENGTH + DPML) / 2   
    
    fixed_geometry = [
        mp.Block(size=mp.Vector3(WG_LENGTH + DPML + 0.1, WG_WIDTH, mp.inf), center=mp.Vector3(input_wg_center_x, 0), material=SI_MEDIUM),
        mp.Block(size=mp.Vector3(WG_LENGTH + DPML + 0.1, WG_WIDTH, mp.inf), center=mp.Vector3(through_wg_center_x, 0), material=SI_MEDIUM),
        mp.Block(size=mp.Vector3(WG_WIDTH, WG_LENGTH + DPML + 0.1, mp.inf), center=mp.Vector3(0, cross_top_center_y), material=SI_MEDIUM),
        mp.Block(size=mp.Vector3(WG_WIDTH, WG_LENGTH + DPML + 0.1, mp.inf), center=mp.Vector3(0, cross_bot_center_y), material=SI_MEDIUM),
    ]
    full_geometry = fixed_geometry + mdm_structure
    
    src_center = mp.Vector3(-sx / 2 + DPML + 0.2, 0)
    src_size = mp.Vector3(0, WG_WIDTH) 
    mon_x_through = mdm_lc / 2 + WG_LENGTH / 2
    monitor_size_y = mp.Vector3(0, WG_WIDTH * 3)
    
    modes = ['TE0', 'TE1']
    
    for mode_name in modes:
        mp.Simulation(cell_size=cell, resolution=1, boundary_layers=[]).reset_meep()
        print(f"  -- 正在模擬 {mode_name} 模態 (100 個波長點) --")
        
        props = {'TE0': {'band': 1, 'parity': mp.EVEN_Y}, 'TE1': {'band': 2, 'parity': mp.ODD_Y}}[mode_name]
        
        sources = [mp.EigenModeSource(src=mp.GaussianSource(FCEN, fwidth=DF), 
                                      center=src_center, size=src_size, direction=mp.X, 
                                      eig_band=props['band'], eig_parity=props['parity'])]
        
        sim = mp.Simulation(cell_size=cell, boundary_layers=[mp.PML(DPML)], 
                            geometry=full_geometry, sources=sources, 
                            resolution=RESOLUTION, default_material=SIO2_MEDIUM)
        
        # 100 個獨立的 Monitor 取樣點
        nfreq = 100
        target_wls = np.linspace(1.52, 1.60, nfreq) 
        target_freqs = [1/wl for wl in target_wls]
        
        in_flux_region = mp.ModeRegion(center=mp.Vector3(src_center.x + 0.5, 0), size=monitor_size_y)
        through_flux_region = mp.ModeRegion(center=mp.Vector3(mon_x_through, 0), size=monitor_size_y)
        
        norm_fluxes = [sim.add_mode_monitor(f, 0, 1, in_flux_region) for f in target_freqs]
        flux_throughs = [sim.add_mode_monitor(f, 0, 1, through_flux_region) for f in target_freqs]
        
        sim.run(until_after_sources=mp.stop_when_fields_decayed(50, mp.Ez, mp.Vector3(mon_x_through, WG_WIDTH/4), 1e-4))
        
        wls = []
        trans_dbs = []
        
        for i in range(nfreq):
            wl = target_wls[i]
            res_input = sim.get_eigenmode_coefficients(norm_fluxes[i], [props['band']], eig_parity=props['parity']).alpha[0, 0, 0]
            input_power = np.abs(res_input)**2 + 1e-12
            
            through_coeff = sim.get_eigenmode_coefficients(flux_throughs[i], [props['band']], eig_parity=props['parity']).alpha[0, 0, 0]
            trans = np.abs(through_coeff)**2 / input_power
            trans_db = 10 * np.log10(trans + 1e-9)
            
            wls.append(wl)
            trans_dbs.append(trans_db)
            
        # 模擬完該模態後，立刻畫圖並存檔！
        plot_single_continuous_spectrum(wls, trans_dbs, mode_name, exp_label, color_hex)
        sim.reset_meep()

# ==============================================================================
# 主程式
# ==============================================================================
if __name__ == "__main__":
    experiments = [
        {
            'label': 'Design Region 1 (2.4 μm)',
            'path': '/home/oscar3102/FMQA/Ultra_compact_waveguide_Crossing/Results_Crossing/Crossing_waveguide_1000_100x30_20260531_015644/best_config_30x30.npy',
            'mdm_lc': 2.4,
            'color': '#1f77b4' # 藍色
        },
        {
            'label': 'Design Region 2 (5.0 μm)',
            'path': '/home/oscar3102/FMQA/Ultra_compact_waveguide_Crossing/Results_Crossing/Crossing_waveguide_1000_100x30_20260529_220817/best_config_20x20.npy',
            'mdm_lc': 5.0, 
            'color': '#d62728' # 紅色
        },
        {
            'label': 'Design Region 3 (4.0 μm)',
            'path': '/home/oscar3102/FMQA/Ultra_compact_waveguide_Crossing/Results_Crossing/Crossing_waveguide_1000_100x30_20260531_001805/best_config_30x30.npy',
            'mdm_lc': 4.0, 
            'color': '#2ca02c' # 綠色
        }
    ]
    
    for exp in experiments:
        if os.path.exists(exp['path']):
            # 依序執行並畫圖
            evaluate_and_plot(exp['path'], exp['mdm_lc'], exp['label'], exp['color'])
        else:
            print(f"❌ 找不到檔案: {exp['path']}")

    print("\n🎉 所有模擬與繪圖已順利完成，共產生 6 張圖表！")
    


  開始處理: Design Region 1 (2.4 μm)
  -- 正在模擬 TE0 模態 (100 個波長點) --
-----------
Initializing structure...
time for choose_chunkdivision = 0.000298023 s
Working in 2D dimensions.
Computational cell is 6.4 x 6.4 x 0 with resolution 40
     block, center = (-2.2,0,0)
          size (2.1,1,1e+20)
          axes (1,0,0), (0,1,0), (0,0,1)
          dielectric constant epsilon diagonal = (12.1104,12.1104,12.1104)
     block, center = (2.2,0,0)
          size (2.1,1,1e+20)
          axes (1,0,0), (0,1,0), (0,0,1)
          dielectric constant epsilon diagonal = (12.1104,12.1104,12.1104)
     block, center = (0,2.2,0)
          size (1,2.1,1e+20)
          axes (1,0,0), (0,1,0), (0,0,1)
          dielectric constant epsilon diagonal = (12.1104,12.1104,12.1104)
     block, center = (0,-2.2,0)
          size (1,2.1,1e+20)
          axes (1,0,0), (0,1,0), (0,0,1)
          dielectric constant epsilon diagonal = (12.1104,12.1104,12.1104)
     block, center = (0,0,0)
          size (2.4,2.4,1e+20)
    